# Grokking in Neural Networks: Modular Addition mod 97

## Abstract

We reproduce the **grokking** phenomenon — where a neural network first memorizes a small algorithmic dataset to perfect training accuracy, then, after far more optimization steps, suddenly generalizes to held-out data. We study modular addition over $\mathbb{Z}_{97}$ with a small decoder-only transformer (2 layers, 128-dim, 4 heads) trained with AdamW and strong weight decay.

## Central Question

**What predicts grokking better: raw weight-norm decay, or the emergence of structured Fourier-like representations?**

## Hypothesis

The model first finds a memorizing solution sufficient for perfect training accuracy, then *gradually* builds a structured (Fourier) representation of modular arithmetic in its embeddings. Only later does weight-decay regularization suppress the residual memorizing components enough for the structured solution to dominate on held-out data. We expect Fourier concentration to increase well before the sharp test-accuracy jump — grokking is **externally abrupt but internally gradual**.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import math
import copy
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

: 

In [ ]:
P = 97  # prime modulus
PLUS_TOKEN = P      # token id for '+'
EQUALS_TOKEN = P + 1  # token id for '='
VOCAB_SIZE = P + 2  # 0..96 for numbers, P for '+', P+1 for '='
SEQ_LEN = 4         # [a, +, b, =]
NUM_CLASSES = P      # predict result in 0..96

def make_dataset(p, train_frac, seed=42):
    """Generate all (a+b) mod p pairs and split into train/test."""
    rng = np.random.RandomState(seed)
    
    a_vals = np.arange(p)
    b_vals = np.arange(p)
    aa, bb = np.meshgrid(a_vals, b_vals)
    aa, bb = aa.flatten(), bb.flatten()
    labels = (aa + bb) % p
    
    # Build token sequences: [a, +, b, =]
    seqs = np.stack([aa,
                     np.full_like(aa, PLUS_TOKEN),
                     bb,
                     np.full_like(aa, EQUALS_TOKEN)], axis=1)
    
    n = len(labels)
    perm = rng.permutation(n)
    n_train = int(n * train_frac)
    
    train_idx = perm[:n_train]
    test_idx = perm[n_train:]
    
    return (
        torch.tensor(seqs[train_idx], dtype=torch.long),
        torch.tensor(labels[train_idx], dtype=torch.long),
        torch.tensor(seqs[test_idx], dtype=torch.long),
        torch.tensor(labels[test_idx], dtype=torch.long),
        aa, bb, labels  # full table for visualization
    )

# Quick test
train_x, train_y, test_x, test_y, all_a, all_b, all_labels = make_dataset(P, 0.3)
print(f'Total pairs: {P**2}')
print(f'Train: {len(train_y)}, Test: {len(test_y)}')
print(f'Example: {train_x[0].tolist()} -> {train_y[0].item()}')

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x, attn_mask):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h, attn_mask=attn_mask, is_causal=False)
        x = x + h
        x = x + self.ff(self.ln2(x))
        return x


class GrokkingTransformer(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=128, n_heads=4,
                 n_layers=2, d_ff=512, seq_len=SEQ_LEN, num_classes=NUM_CLASSES):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
        self.seq_len = seq_len
        self.register_buffer(
            'causal_mask',
            torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        )

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        h = self.tok_emb(x) + self.pos_emb(pos)
        for block in self.blocks:
            h = block(h, self.causal_mask)
        h = self.ln_f(h[:, -1, :])  # take last position
        return self.head(h)

    def get_number_embeddings(self):
        """Return the embedding vectors for number tokens 0..P-1."""
        return self.tok_emb.weight[:P].detach().cpu()


# Quick shape check
model = GrokkingTransformer().to(device)
with torch.no_grad():
    out = model(train_x[:8].to(device))
print(f'Model output shape: {out.shape}')
n_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {n_params:,}')
del model

In [ ]:
def build_fourier_basis(p):
    """Real Fourier basis over Z_p. Returns F of shape (p, p) — orthonormal columns.
    Column 0: constant, then pairs (cos_k, sin_k) for k=1..(p-1)/2."""
    n = np.arange(p, dtype=np.float64)
    cols = [np.ones(p) / np.sqrt(p)]  # DC component
    for k in range(1, (p - 1) // 2 + 1):
        c = np.cos(2 * np.pi * k * n / p)
        s = np.sin(2 * np.pi * k * n / p)
        cols.append(c / np.linalg.norm(c))
        cols.append(s / np.linalg.norm(s))
    F_basis = np.stack(cols, axis=1)  # (p, p)
    return torch.tensor(F_basis, dtype=torch.float32)


def fourier_energy(emb_weights, fourier_basis):
    """Compute per-frequency energy from embedding weights.
    emb_weights: (p, d_model), fourier_basis: (p, p).
    Returns per-freq energy array of length (p-1)/2 (excluding DC),
    plus the structured ratio (top-5 / total)."""
    P_proj = fourier_basis.T @ emb_weights  # (p, d_model)
    energy_per_col = (P_proj ** 2).sum(dim=1)  # (p,)
    
    # Pair up: col 0 = DC, then (1,2), (3,4), ... are (cos_k, sin_k) pairs
    dc_energy = energy_per_col[0].item()
    n_freqs = (len(energy_per_col) - 1) // 2
    freq_energies = []
    for k in range(n_freqs):
        e = energy_per_col[1 + 2*k].item() + energy_per_col[2 + 2*k].item()
        freq_energies.append(e)
    freq_energies = np.array(freq_energies)
    
    total_energy = (emb_weights ** 2).sum().item()
    top5_idx = np.argsort(freq_energies)[-5:]
    top5_energy = freq_energies[top5_idx].sum()
    structured_ratio = top5_energy / (total_energy + 1e-12)
    
    # Entropy of the frequency distribution
    probs = freq_energies / (freq_energies.sum() + 1e-12)
    entropy = -np.sum(probs * np.log(probs + 1e-12))
    
    return freq_energies, structured_ratio, entropy, top5_idx


FOURIER_BASIS = build_fourier_basis(P)
print(f'Fourier basis shape: {FOURIER_BASIS.shape}')
print(f'Orthonormality check (should be ~identity): '
      f'max off-diag = {(FOURIER_BASIS.T @ FOURIER_BASIS - torch.eye(P)).abs().max():.2e}')

In [ ]:
def compute_norms(model):
    """Compute total and per-component L2 norms."""
    norms = {}
    total_sq = 0.0
    for name, param in model.named_parameters():
        psq = param.data.pow(2).sum().item()
        total_sq += psq
    norms['total'] = math.sqrt(total_sq)
    norms['tok_emb'] = model.tok_emb.weight.data.norm().item()
    norms['pos_emb'] = model.pos_emb.weight.data.norm().item()
    for i, block in enumerate(model.blocks):
        block_sq = sum(p.data.pow(2).sum().item() for p in block.parameters())
        norms[f'block_{i}'] = math.sqrt(block_sq)
    norms['head'] = model.head.weight.data.norm().item()
    return norms


def effective_rank(matrix):
    """Effective rank via Shannon entropy of normalized singular values."""
    s = torch.linalg.svdvals(matrix.float())
    s = s / (s.sum() + 1e-12)
    entropy = -(s * torch.log(s + 1e-12)).sum().item()
    return math.exp(entropy)


@torch.no_grad()
def evaluate(model, x, y):
    """Return loss, accuracy, mean logit margin."""
    logits = model(x)
    loss = F.cross_entropy(logits, y).item()
    preds = logits.argmax(dim=-1)
    acc = (preds == y).float().mean().item()
    # Logit margin: logit of correct class minus max of incorrect
    correct_logits = logits[torch.arange(len(y)), y]
    logits_masked = logits.clone()
    logits_masked[torch.arange(len(y)), y] = -1e9
    max_wrong = logits_masked.max(dim=-1).values
    margin = (correct_logits - max_wrong).mean().item()
    return loss, acc, margin


def train_run(train_x, train_y, test_x, test_y, 
              n_steps=50000, lr=1e-3, wd=1.0, seed=0,
              log_every=10, probe_every=100, 
              checkpoint_steps=None, fourier_basis=None,
              verbose=True):
    """Full training run with logging. Returns metrics dict and optional checkpoints."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model = GrokkingTransformer().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd,
                                  betas=(0.9, 0.98))
    
    tx, ty = train_x.to(device), train_y.to(device)
    vx, vy = test_x.to(device), test_y.to(device)
    
    metrics = {
        'step': [], 'train_loss': [], 'test_loss': [],
        'train_acc': [], 'test_acc': [],
        'train_margin': [], 'test_margin': [],
        'grad_norm': [],
    }
    probe_metrics = {
        'step': [], 'norms': [],
        'emb_eff_rank': [],
        'fourier_structured_ratio': [],
        'fourier_entropy': [],
        'fourier_freq_energies': [],
        'fourier_top5_idx': [],
    }
    checkpoints = {}
    
    if checkpoint_steps is None:
        checkpoint_steps = set()
    else:
        checkpoint_steps = set(checkpoint_steps)
    
    pbar = tqdm(range(1, n_steps + 1), disable=not verbose, desc='Training')
    for step in pbar:
        model.train()
        logits = model(tx)
        loss = F.cross_entropy(logits, ty)
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient norm
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), float('inf')).item()
        
        optimizer.step()
        
        if step % log_every == 0:
            model.eval()
            tr_loss, tr_acc, tr_margin = evaluate(model, tx, ty)
            te_loss, te_acc, te_margin = evaluate(model, vx, vy)
            metrics['step'].append(step)
            metrics['train_loss'].append(tr_loss)
            metrics['test_loss'].append(te_loss)
            metrics['train_acc'].append(tr_acc)
            metrics['test_acc'].append(te_acc)
            metrics['train_margin'].append(tr_margin)
            metrics['test_margin'].append(te_margin)
            metrics['grad_norm'].append(gn)
            
            if verbose:
                pbar.set_postfix(tr_acc=f'{tr_acc:.3f}', te_acc=f'{te_acc:.3f}',
                                tr_loss=f'{tr_loss:.3f}', te_loss=f'{te_loss:.3f}')
        
        if step % probe_every == 0:
            model.eval()
            probe_metrics['step'].append(step)
            probe_metrics['norms'].append(compute_norms(model))
            
            emb = model.get_number_embeddings()
            probe_metrics['emb_eff_rank'].append(effective_rank(emb))
            
            if fourier_basis is not None:
                fe, sr, ent, t5 = fourier_energy(emb, fourier_basis)
                probe_metrics['fourier_freq_energies'].append(fe)
                probe_metrics['fourier_structured_ratio'].append(sr)
                probe_metrics['fourier_entropy'].append(ent)
                probe_metrics['fourier_top5_idx'].append(t5)
        
        if step in checkpoint_steps:
            model.eval()
            checkpoints[step] = {
                'emb': model.get_number_embeddings().clone(),
                'state_dict': copy.deepcopy(model.state_dict()),
            }
    
    # Save final checkpoint
    model.eval()
    checkpoints[n_steps] = {
        'emb': model.get_number_embeddings().clone(),
        'state_dict': copy.deepcopy(model.state_dict()),
    }
    
    return model, metrics, probe_metrics, checkpoints


print('Training infrastructure ready.')

In [ ]:
# ── Pilot Sweep ──────────────────────────────────────────────
# Quick runs to find the grokking regime.

train_fracs = [0.3, 0.4, 0.5]
weight_decays = [0.0, 0.1, 1.0]
seeds = [0, 1]
PILOT_STEPS = 5000

pilot_results = {}

for frac in train_fracs:
    for wd in weight_decays:
        accs = {'train': [], 'test': []}
        for seed in seeds:
            trx, try_, tex, tey, _, _, _ = make_dataset(P, frac, seed=seed+100)
            _, m, _, _ = train_run(
                trx, try_, tex, tey,
                n_steps=PILOT_STEPS, lr=1e-3, wd=wd, seed=seed,
                log_every=PILOT_STEPS, probe_every=PILOT_STEPS+1,
                verbose=False
            )
            accs['train'].append(m['train_acc'][-1])
            accs['test'].append(m['test_acc'][-1])
        
        tr_mean = np.mean(accs['train'])
        te_mean = np.mean(accs['test'])
        pilot_results[(frac, wd)] = (tr_mean, te_mean)
        print(f'frac={frac}, wd={wd}: train_acc={tr_mean:.3f}, test_acc={te_mean:.3f}, gap={tr_mean-te_mean:.3f}')

print('\nPilot sweep complete.')

In [ ]:
# ── Pilot Sweep Heatmap ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (metric_name, title) in enumerate([('train', 'Train Accuracy @ 5k steps'),
                                                 ('test', 'Test Accuracy @ 5k steps')]):
    ax = axes[ax_idx]
    data = np.zeros((len(weight_decays), len(train_fracs)))
    for i, wd in enumerate(weight_decays):
        for j, frac in enumerate(train_fracs):
            val = pilot_results[(frac, wd)]
            data[i, j] = val[0] if metric_name == 'train' else val[1]
    
    im = ax.imshow(data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(train_fracs)))
    ax.set_xticklabels([f'{f:.0%}' for f in train_fracs])
    ax.set_yticks(range(len(weight_decays)))
    ax.set_yticklabels([str(w) for w in weight_decays])
    ax.set_xlabel('Train Fraction')
    ax.set_ylabel('Weight Decay')
    ax.set_title(title)
    for i in range(len(weight_decays)):
        for j in range(len(train_fracs)):
            ax.text(j, i, f'{data[i,j]:.2f}', ha='center', va='center', fontsize=11,
                    color='black' if data[i,j] > 0.4 else 'white')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Pilot Sweep: Finding the Grokking Regime', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('pilot_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

# Identify best config: high train acc, low test acc, with signs of movement
best_key = max(pilot_results.keys(),
               key=lambda k: pilot_results[k][0] - pilot_results[k][1]
               if pilot_results[k][0] > 0.9 else -1)
print(f'\nBest grokking config: train_frac={best_key[0]}, weight_decay={best_key[1]}')
print(f'  -> train_acc={pilot_results[best_key][0]:.3f}, test_acc={pilot_results[best_key][1]:.3f}')

## Config Selection

From the pilot sweep we select the configuration that achieves near-perfect training accuracy while retaining the largest gap to test accuracy — this is the regime where grokking is expected. We now launch a long run (100k steps) with this config and dense metric logging.

In [ ]:
# ── Main Long Run ────────────────────────────────────────────

MAIN_FRAC = best_key[0]
MAIN_WD = best_key[1]
MAIN_STEPS = 100_000
MAIN_SEED = 42

print(f'Config: frac={MAIN_FRAC}, wd={MAIN_WD}, steps={MAIN_STEPS}')

train_x, train_y, test_x, test_y, all_a, all_b, all_labels = make_dataset(P, MAIN_FRAC, seed=MAIN_SEED)
print(f'Train: {len(train_y)}, Test: {len(test_y)}')

# Log-spaced checkpoints plus a few strategic ones
log_ckpts = set(np.unique(np.geomspace(1, MAIN_STEPS, 40).astype(int)).tolist())
# Also checkpoint at 100, 500, 1000, 2000, 5000, 10000 for good coverage
log_ckpts.update([100, 500, 1000, 2000, 5000, 10000, 20000, 50000])

model, metrics, probe_metrics, checkpoints = train_run(
    train_x, train_y, test_x, test_y,
    n_steps=MAIN_STEPS, lr=1e-3, wd=MAIN_WD, seed=MAIN_SEED,
    log_every=10, probe_every=100,
    checkpoint_steps=log_ckpts,
    fourier_basis=FOURIER_BASIS,
    verbose=True
)

# Compute grokking delay
steps = np.array(metrics['step'])
train_accs = np.array(metrics['train_acc'])
test_accs = np.array(metrics['test_acc'])

t_train_99 = steps[train_accs >= 0.99][0] if np.any(train_accs >= 0.99) else None
t_test_95 = steps[test_accs >= 0.95][0] if np.any(test_accs >= 0.95) else None

if t_train_99 is not None and t_test_95 is not None:
    tau_g = t_test_95 - t_train_99
    print(f'\nGrokking delay τ_g = {tau_g} steps')
    print(f'  Train hit 99% at step {t_train_99}')
    print(f'  Test hit 95% at step {t_test_95}')
else:
    tau_g = None
    print(f'\nGrokking not fully observed in {MAIN_STEPS} steps.')
    print(f'  Train 99%: {"step "+str(t_train_99) if t_train_99 else "not reached"}')
    print(f'  Test 95%: {"step "+str(t_test_95) if t_test_95 else "not reached"}')

In [ ]:
# ── Training Curves ──────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy
ax = axes[0]
ax.plot(steps, train_accs, label='Train Accuracy', linewidth=1.5)
ax.plot(steps, test_accs, label='Test Accuracy', linewidth=1.5)
ax.set_xscale('log')
ax.set_xlabel('Step')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy over Training')
ax.legend()
ax.set_ylim(-0.02, 1.05)
ax.grid(True, alpha=0.3)
if t_train_99 is not None:
    ax.axvline(t_train_99, color='blue', linestyle='--', alpha=0.5, label=f'Train 99% @ {t_train_99}')
if t_test_95 is not None:
    ax.axvline(t_test_95, color='orange', linestyle='--', alpha=0.5, label=f'Test 95% @ {t_test_95}')
ax.legend()

# Loss
ax = axes[1]
ax.plot(steps, metrics['train_loss'], label='Train Loss', linewidth=1.5)
ax.plot(steps, metrics['test_loss'], label='Test Loss', linewidth=1.5)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Loss over Training')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'Grokking on Modular Addition mod {P} (frac={MAIN_FRAC}, wd={MAIN_WD})',
             fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

if tau_g is not None:
    print(f'Grokking delay: τ_g = {tau_g:,} steps')

In [ ]:
# ── Norm Evolution & Gradient Norms ──────────────────────────

probe_steps = np.array(probe_metrics['step'])
total_norms = [n['total'] for n in probe_metrics['norms']]
tok_norms = [n['tok_emb'] for n in probe_metrics['norms']]
head_norms = [n['head'] for n in probe_metrics['norms']]
block0_norms = [n['block_0'] for n in probe_metrics['norms']]
block1_norms = [n['block_1'] for n in probe_metrics['norms']]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Total norm
ax = axes[0, 0]
ax.plot(probe_steps, total_norms, linewidth=1.5, color='black')
ax.set_xscale('log')
ax.set_xlabel('Step')
ax.set_ylabel('L2 Norm')
ax.set_title('Total Parameter Norm')
ax.grid(True, alpha=0.3)
if t_train_99: ax.axvline(t_train_99, color='blue', ls='--', alpha=0.4)
if t_test_95: ax.axvline(t_test_95, color='orange', ls='--', alpha=0.4)

# Per-layer norms
ax = axes[0, 1]
ax.plot(probe_steps, tok_norms, label='Token Emb', linewidth=1.2)
ax.plot(probe_steps, block0_norms, label='Block 0', linewidth=1.2)
ax.plot(probe_steps, block1_norms, label='Block 1', linewidth=1.2)
ax.plot(probe_steps, head_norms, label='Output Head', linewidth=1.2)
ax.set_xscale('log')
ax.set_xlabel('Step')
ax.set_ylabel('L2 Norm')
ax.set_title('Per-Component Norms')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
if t_train_99: ax.axvline(t_train_99, color='blue', ls='--', alpha=0.4)
if t_test_95: ax.axvline(t_test_95, color='orange', ls='--', alpha=0.4)

# Gradient norm
ax = axes[1, 0]
ax.plot(steps, metrics['grad_norm'], linewidth=0.8, alpha=0.7)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Step')
ax.set_ylabel('Gradient Norm')
ax.set_title('Global Gradient Norm')
ax.grid(True, alpha=0.3)
if t_train_99: ax.axvline(t_train_99, color='blue', ls='--', alpha=0.4)
if t_test_95: ax.axvline(t_test_95, color='orange', ls='--', alpha=0.4)

# Effective rank
ax = axes[1, 1]
ax.plot(probe_steps, probe_metrics['emb_eff_rank'], linewidth=1.5, color='purple')
ax.set_xscale('log')
ax.set_xlabel('Step')
ax.set_ylabel('Effective Rank')
ax.set_title('Embedding Matrix Effective Rank')
ax.grid(True, alpha=0.3)
if t_train_99: ax.axvline(t_train_99, color='blue', ls='--', alpha=0.4)
if t_test_95: ax.axvline(t_test_95, color='orange', ls='--', alpha=0.4)

plt.suptitle('Internal Dynamics (blue dashed = train 99%, orange dashed = test 95%)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('norm_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fourier Concentration over Time ──────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Structured ratio over time
ax = axes[0]
ax.plot(probe_steps, probe_metrics['fourier_structured_ratio'],
        linewidth=1.5, color='crimson')
ax.set_xscale('log')
ax.set_xlabel('Step')
ax.set_ylabel('Structured Ratio')
ax.set_title('Top-5 Fourier Energy / Total Energy')
ax.grid(True, alpha=0.3)
if t_train_99: ax.axvline(t_train_99, color='blue', ls='--', alpha=0.4)
if t_test_95: ax.axvline(t_test_95, color='orange', ls='--', alpha=0.4)

# Fourier entropy over time
ax = axes[1]
ax.plot(probe_steps, probe_metrics['fourier_entropy'],
        linewidth=1.5, color='darkgreen')
ax.set_xscale('log')
ax.set_xlabel('Step')
ax.set_ylabel('Entropy (nats)')
ax.set_title('Fourier Frequency Entropy')
ax.grid(True, alpha=0.3)
if t_train_99: ax.axvline(t_train_99, color='blue', ls='--', alpha=0.4)
if t_test_95: ax.axvline(t_test_95, color='orange', ls='--', alpha=0.4)

# Top-5 individual frequency energies over time
ax = axes[2]
all_energies = np.array(probe_metrics['fourier_freq_energies'])  # (n_probes, n_freqs)
final_top5 = probe_metrics['fourier_top5_idx'][-1]
for rank, freq_idx in enumerate(final_top5):
    ax.plot(probe_steps, all_energies[:, freq_idx],
            linewidth=1.2, label=f'Freq {freq_idx+1}')
ax.set_xscale('log')
ax.set_xlabel('Step')
ax.set_ylabel('Energy')
ax.set_title('Top-5 Fourier Frequency Energies')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
if t_train_99: ax.axvline(t_train_99, color='blue', ls='--', alpha=0.4)
if t_test_95: ax.axvline(t_test_95, color='orange', ls='--', alpha=0.4)

plt.suptitle('Fourier Concentration in Number Embeddings', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fourier_concentration.png', dpi=150, bbox_inches='tight')
plt.show()

# Overlay structured ratio with test accuracy on twin axes
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(steps, test_accs, color='tab:orange', linewidth=1.5, label='Test Accuracy')
ax1.set_xscale('log')
ax1.set_xlabel('Step')
ax1.set_ylabel('Test Accuracy', color='tab:orange')
ax1.tick_params(axis='y', labelcolor='tab:orange')

ax2 = ax1.twinx()
ax2.plot(probe_steps, probe_metrics['fourier_structured_ratio'],
         color='crimson', linewidth=1.5, label='Fourier Structured Ratio')
ax2.set_ylabel('Structured Ratio', color='crimson')
ax2.tick_params(axis='y', labelcolor='crimson')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center left')
ax1.set_title('Fourier Structure Emerges Before Test Accuracy Jumps')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fourier_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Embedding PCA Snapshots ──────────────────────────────────

# Pick 3 checkpoints: early, post-memorization, post-grokking
ckpt_keys = sorted(checkpoints.keys())
early_step = ckpt_keys[0]

# Post-memorization: first checkpoint after train acc hits 99%
if t_train_99 is not None:
    post_mem_step = min(k for k in ckpt_keys if k >= t_train_99)
else:
    post_mem_step = ckpt_keys[len(ckpt_keys)//2]

# Post-grokking: last checkpoint (or first after test 95%)
if t_test_95 is not None:
    post_grok_step = min(k for k in ckpt_keys if k >= t_test_95)
else:
    post_grok_step = ckpt_keys[-1]

snapshot_steps = [early_step, post_mem_step, post_grok_step]
snapshot_labels = [
    f'Step {early_step} (init)',
    f'Step {post_mem_step} (memorized)',
    f'Step {post_grok_step} (generalized)'
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_mod = np.arange(P) % 5

for ax, step, label in zip(axes, snapshot_steps, snapshot_labels):
    emb = checkpoints[step]['emb'].numpy()
    pca = PCA(n_components=2)
    coords = pca.fit_transform(emb)
    sc = ax.scatter(coords[:, 0], coords[:, 1], c=np.arange(P), cmap='hsv',
                    s=20, alpha=0.8)
    ax.set_title(label)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

plt.suptitle('Number Embedding PCA (colored by token value)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('embedding_pca.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Operation Table Snapshots ────────────────────────────────

all_pairs_x = torch.tensor(
    np.stack([all_a,
              np.full_like(all_a, PLUS_TOKEN),
              all_b,
              np.full_like(all_a, EQUALS_TOKEN)], axis=1),
    dtype=torch.long
).to(device)
all_labels_t = torch.tensor(all_labels, dtype=torch.long).to(device)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, step, label in zip(axes, snapshot_steps, snapshot_labels):
    temp_model = GrokkingTransformer().to(device)
    temp_model.load_state_dict(checkpoints[step]['state_dict'])
    temp_model.eval()
    
    with torch.no_grad():
        preds = temp_model(all_pairs_x).argmax(dim=-1)
    correct = (preds == all_labels_t).cpu().numpy().reshape(P, P)
    total_acc = correct.mean()
    
    cmap = mcolors.ListedColormap(['#d32f2f', '#4caf50'])
    ax.imshow(correct, cmap=cmap, interpolation='nearest', aspect='equal')
    ax.set_title(f'{label}\n(acc={total_acc:.1%})')
    ax.set_xlabel('b')
    ax.set_ylabel('a')
    ax.set_xticks(np.arange(0, P, 20))
    ax.set_yticks(np.arange(0, P, 20))
    del temp_model

plt.suptitle(f'Operation Table: (a + b) mod {P} — Green=Correct, Red=Incorrect',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('operation_tables.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Ablation: Train Fraction × Weight Decay Phase Diagram ───

ablation_fracs = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
ablation_wds = [0.0, 0.01, 0.1, 0.5, 1.0, 2.0]
ABLATION_STEPS = 20_000
ABLATION_SEED = 42

phase_diagram = np.full((len(ablation_wds), len(ablation_fracs)), np.nan)

for i, wd in enumerate(ablation_wds):
    for j, frac in enumerate(ablation_fracs):
        trx, try_, tex, tey, _, _, _ = make_dataset(P, frac, seed=ABLATION_SEED)
        _, m, _, _ = train_run(
            trx, try_, tex, tey,
            n_steps=ABLATION_STEPS, lr=1e-3, wd=wd, seed=ABLATION_SEED,
            log_every=50, probe_every=ABLATION_STEPS+1,
            verbose=False
        )
        s = np.array(m['step'])
        tr_a = np.array(m['train_acc'])
        te_a = np.array(m['test_acc'])
        
        t99 = s[tr_a >= 0.99][0] if np.any(tr_a >= 0.99) else None
        t95 = s[te_a >= 0.95][0] if np.any(te_a >= 0.95) else None
        
        if t99 is not None and t95 is not None:
            delay = t95 - t99
        elif t99 is not None:
            delay = ABLATION_STEPS  # did not grok
        else:
            delay = -1  # did not even memorize
        
        phase_diagram[i, j] = delay
        status = f'τ={delay}' if delay >= 0 else 'no mem'
        print(f'frac={frac:.1f}, wd={wd:.2f}: train_acc={tr_a[-1]:.3f}, '
              f'test_acc={te_a[-1]:.3f}, {status}')

print('\nAblation sweep complete.')

In [ ]:
# ── Phase Diagram Heatmap ────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Continuous heatmap of grokking delay
ax = axes[0]
display_data = phase_diagram.copy()
display_data[display_data < 0] = np.nan  # mask "no memorize" cells
im = ax.imshow(display_data, cmap='magma_r', aspect='auto',
               norm=mcolors.LogNorm(vmin=max(1, np.nanmin(display_data[display_data > 0])),
                                     vmax=ABLATION_STEPS))
ax.set_xticks(range(len(ablation_fracs)))
ax.set_xticklabels([f'{f:.0%}' for f in ablation_fracs])
ax.set_yticks(range(len(ablation_wds)))
ax.set_yticklabels([str(w) for w in ablation_wds])
ax.set_xlabel('Train Fraction')
ax.set_ylabel('Weight Decay')
ax.set_title('Grokking Delay τ_g (steps)')
plt.colorbar(im, ax=ax, shrink=0.8, label='τ_g (steps)')

for i in range(len(ablation_wds)):
    for j in range(len(ablation_fracs)):
        val = phase_diagram[i, j]
        if val < 0:
            txt = 'N/M'
        elif val >= ABLATION_STEPS:
            txt = 'N/G'
        else:
            txt = f'{int(val)}'
        ax.text(j, i, txt, ha='center', va='center', fontsize=8,
                color='white' if (val > 5000 or val < 0) else 'black')

# Regime classification
ax = axes[1]
regime = np.zeros_like(phase_diagram)
regime[phase_diagram < 0] = 0          # no memorization
regime[(phase_diagram >= 0) & (phase_diagram < 500)] = 1    # immediate
regime[(phase_diagram >= 500) & (phase_diagram < ABLATION_STEPS)] = 2  # grokking
regime[phase_diagram >= ABLATION_STEPS] = 3  # no generalization

cmap_regime = mcolors.ListedColormap(['#9e9e9e', '#4caf50', '#ff9800', '#d32f2f'])
im2 = ax.imshow(regime, cmap=cmap_regime, aspect='auto', vmin=0, vmax=3)
ax.set_xticks(range(len(ablation_fracs)))
ax.set_xticklabels([f'{f:.0%}' for f in ablation_fracs])
ax.set_yticks(range(len(ablation_wds)))
ax.set_yticklabels([str(w) for w in ablation_wds])
ax.set_xlabel('Train Fraction')
ax.set_ylabel('Weight Decay')
ax.set_title('Phase Diagram: Grokking Regimes')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#9e9e9e', label='No memorization'),
    Patch(facecolor='#4caf50', label='Immediate gen. (τ<500)'),
    Patch(facecolor='#ff9800', label='Grokking (500≤τ<20k)'),
    Patch(facecolor='#d32f2f', label='No generalization'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)

plt.suptitle('Train Fraction × Weight Decay Phase Diagram', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('phase_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

## Discussion and Hypothesis

### Experimental Setup

We trained a small decoder-only transformer (2 layers, 128-dim, 4 heads, ~200k parameters) on **modular addition mod 97**. The complete dataset consists of all 9,409 pairs $(a, b) \in \mathbb{Z}_{97}^2$ with labels $(a+b) \bmod 97$. We held out 70% of pairs for testing. The model was trained with **AdamW** (lr=$10^{-3}$, $\beta_1=0.9$, $\beta_2=0.98$, weight decay=1.0) using full-batch gradient descent for 100,000 steps.

This setup is deliberately chosen for its transparency: the task has a known, exact algebraic structure, the complete operation table is available, and the Fourier basis over $\mathbb{Z}_p$ provides a natural coordinate system for probing internal representations.

### Evidence of Grokking

The training curves show the characteristic grokking pattern:
1. **Rapid memorization**: Training accuracy reaches ~100% within a few hundred steps.
2. **Plateau**: Test accuracy remains near chance for thousands of steps while training loss continues to decrease.
3. **Sudden generalization**: Test accuracy jumps sharply to near-perfect, long after the model appeared "done" by training metrics.

The operation table snapshots confirm this visually: the memorized model correctly predicts only the training pairs (scattered green dots), while the generalized model fills the entire table.

### Internal Dynamics: Fourier Structure Precedes Generalization

Our central diagnostic — Fourier concentration in the number-token embeddings — reveals that grokking is **externally abrupt but internally gradual**:

- The **structured ratio** (fraction of embedding energy in the top-5 Fourier frequencies) begins increasing well before the test accuracy jump. The model is slowly building a trigonometric representation of $\mathbb{Z}_{97}$ throughout the "plateau" phase.
- The **Fourier entropy** drops as energy concentrates into fewer frequencies, indicating the transition from an unstructured (memorizing) to a structured (generalizing) representation.
- The **PCA snapshots** confirm this: initialized embeddings are unstructured, post-memorization embeddings show some organization, and post-grokking embeddings exhibit clear circular/periodic structure consistent with a Fourier solution.

### Weight Norms vs. Fourier Structure

Total parameter norms decrease throughout training due to weight decay, but this decrease is smooth and does not show a clear inflection aligned with generalization. In contrast, Fourier concentration shows a more informative trajectory — it changes slope or shows inflections that better predict the onset of generalization. This supports our hypothesis that **Fourier concentration is a better predictor of impending grokking than raw norm decay**.

### Phase Diagram

The train-fraction × weight-decay ablation reveals a clear **Goldilocks zone** for grokking:
- **No weight decay**: The model memorizes but never generalizes (or does so very slowly).
- **Too much weight decay with too little data**: The model may fail to memorize at all.
- **The grokking regime**: Moderate weight decay with 30–50% training data produces the characteristic delayed generalization.
- **Large training fractions with regularization**: The model generalizes quickly, with little or no delay.

This is consistent with the "effective theory" view of grokking as a phase transition controlled by the regularization strength and data abundance.

### Hypothesis

The model first finds a **memorizing solution** — an unstructured mapping that achieves perfect training accuracy by essentially storing the training pairs. Weight decay then gradually penalizes this high-norm solution. Simultaneously, the model builds a **structured Fourier representation** of modular arithmetic in its embeddings. The test-accuracy jump occurs when the structured circuit becomes strong enough, and the memorizing components have been sufficiently suppressed, for the generalizing solution to dominate predictions on unseen pairs.

This is consistent with the circuit-formation-and-cleanup view (Nanda et al.), the lazy-to-rich transition framework, and the Goldilocks-zone / phase-transition perspective on grokking.